NYISO Webscraper

In [1]:
import requests
from bs4 import BeautifulSoup
import os
import zipfile
from tqdm import tqdm  # pip install tqdm
from urllib.parse import urljoin

# URL of NYISO archived files page
url = "https://mis.nyiso.com/public/P-58Blist.htm"

# fetch page
r = requests.get(url)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

# folder for final CSVs
extract_dir = "nyiso_csvs"
os.makedirs(extract_dir, exist_ok=True)

# find the single table
table = soup.find("table")

# collect all archive links
links = []
collect = False
for tr in table.find_all("tr"):
    text = tr.get_text(strip=True)
    if "Archived Files" in text:
        collect = True
        continue
    if collect:
        link = tr.find("a")
        if link and link.get("href"):
            file_url = urljoin(url, link["href"])  # ✅ safer join
            links.append(file_url)

# download + unzip + delete
for file_url in tqdm(links, desc="Downloading & extracting"):
    file_name = os.path.basename(file_url)
    zip_path = os.path.join(extract_dir, file_name)

    # download
    resp = requests.get(file_url, stream=True)
    resp.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    # extract and delete
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        os.remove(zip_path)  # cleanup zip
    except zipfile.BadZipFile:
        print(f"⚠ Skipping {file_name}, not a valid ZIP.")

print(f"✅ Done! All CSVs are in: {extract_dir}/")

✅ Done! All CSVs are in: nyiso_csvs/


Name the csvs

In [2]:
import os
import csv
from datetime import datetime
import pandas as pd

def get_date_from_csv(csv_path):
    """Extract the first and last dates from a CSV file."""
    try:
        # Read just the first few rows to get the first date
        with open(csv_path, 'r', encoding='utf-8') as file:
            reader = csv.reader(file)
            header = next(reader)  # Skip header
            first_row = next(reader)
            first_timestamp = first_row[0]
        
        # Read the last few rows to get the last date
        # For efficiency, we'll assume single-day files and just use the first date
        # Parse the timestamp format "MM/DD/YYYY HH:MM:SS"
        date_part = first_timestamp.split(' ')[0]  # Get just the date part
        return date_part, date_part  # Return same date for both first and last
        
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        return None, None

def rename_csv_files(directory):
    """Rename all CSV files in the directory based on their internal dates."""
    
    # Get all CSV files in the directory
    csv_files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    
    print(f"Found {len(csv_files)} CSV files to process...")
    
    renamed_count = 0
    errors = []
    
    for csv_file in csv_files:
        csv_path = os.path.join(directory, csv_file)
        
        # Get the dates from the CSV
        first_date, last_date = get_date_from_csv(csv_path)
        
        if first_date and last_date:
            # Since files contain single-day data, first_date equals last_date
            # Format: MM/DD/YYYY -> MM_DD_YYYY for filename (replace / with _)
            formatted_date = first_date.replace('/', '_')
            new_filename = f"{formatted_date}.csv"
            new_path = os.path.join(directory, new_filename)
            
            # Check if file already exists with the new name
            if os.path.exists(new_path) and new_path != csv_path:
                print(f"Warning: {new_filename} already exists, skipping {csv_file}")
                continue
            
            # Rename the file
            try:
                if csv_path != new_path:  # Only rename if different
                    os.rename(csv_path, new_path)
                    print(f"Renamed: {csv_file} -> {new_filename}")
                    renamed_count += 1
                else:
                    print(f"Already correctly named: {csv_file}")
            except Exception as e:
                error_msg = f"Error renaming {csv_file}: {e}"
                print(error_msg)
                errors.append(error_msg)
        else:
            error_msg = f"Could not extract dates from {csv_file}"
            print(error_msg)
            errors.append(error_msg)
    
    print(f"\nRenaming complete!")
    print(f"Successfully renamed: {renamed_count} files")
    if errors:
        print(f"Errors encountered: {len(errors)}")
        for error in errors[:5]:  # Show first 5 errors
            print(f"  - {error}")
        if len(errors) > 5:
            print(f"  ... and {len(errors) - 5} more errors")

if __name__ == "__main__":
    # Directory containing the CSV files
    csv_directory = r"c:\Users\Matt\Desktop\CS506\CS506_Project\nyiso_csvs"
    
    # Confirm directory exists
    if not os.path.exists(csv_directory):
        print(f"Error: Directory {csv_directory} does not exist!")
    else:
        print(f"Processing CSV files in: {csv_directory}")
        rename_csv_files(csv_directory)

Processing CSV files in: c:\Users\Matt\Desktop\CS506\CS506_Project\nyiso_csvs
Found 8864 CSV files to process...
Already correctly named: 01_01_2002.csv
Already correctly named: 01_01_2003.csv
Already correctly named: 01_01_2004.csv
Already correctly named: 01_01_2005.csv
Already correctly named: 01_01_2006.csv
Already correctly named: 01_01_2007.csv
Already correctly named: 01_01_2008.csv
Already correctly named: 01_01_2009.csv
Already correctly named: 01_01_2010.csv
Already correctly named: 01_01_2011.csv
Already correctly named: 01_01_2012.csv
Already correctly named: 01_01_2013.csv
Already correctly named: 01_01_2014.csv
Already correctly named: 01_01_2015.csv
Already correctly named: 01_01_2016.csv
Already correctly named: 01_01_2017.csv
Already correctly named: 01_01_2018.csv
Already correctly named: 01_01_2019.csv
Already correctly named: 01_02_2002.csv
Already correctly named: 01_02_2003.csv
Already correctly named: 01_02_2004.csv
Already correctly named: 01_02_2005.csv
Already

In [ ]:
Sort CSV files by year 

In [6]:
import os
import shutil
from datetime import datetime
import pandas as pd

def sort_csvs_by_year():
    """Sort CSV files from nyiso_csvs into yearly folders in nyiso_yearly."""
    
    # Define source and destination directories
    source_dir = os.path.join("LIB", "nyiso_csvs")
    dest_dir = os.path.join("LIB", "nyiso_csvs")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory {source_dir} does not exist!")
        return
    
    # Get all CSV files
    csv_files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]
    print(f"Found {len(csv_files)} CSV files to sort...")
    
    # Dictionary to track files by year
    files_by_year = {}
    errors = []
    
    for csv_file in csv_files:
        try:
            # Extract year from filename (format: MM_DD_YYYY.csv)
            filename_without_ext = csv_file.replace('.csv', '')
            date_parts = filename_without_ext.split('_')
            
            if len(date_parts) == 3:
                month, day, year = date_parts
                year = int(year)
                
                # Add to tracking dictionary
                if year not in files_by_year:
                    files_by_year[year] = []
                files_by_year[year].append(csv_file)
                
            else:
                errors.append(f"Could not parse date from filename: {csv_file}")
                
        except Exception as e:
            errors.append(f"Error processing {csv_file}: {e}")
    
    # Sort years and process files
    sorted_years = sorted(files_by_year.keys())
    total_moved = 0
    
    for year in sorted_years:
        year_dir = os.path.join(dest_dir, str(year))
        os.makedirs(year_dir, exist_ok=True)
        
        # Sort files within the year by date
        year_files = files_by_year[year]
        year_files.sort(key=lambda x: datetime.strptime(x.replace('.csv', '').replace('_', '/'), '%m/%d/%Y'))
        
        print(f"\nProcessing year {year} ({len(year_files)} files):")
        
        for csv_file in year_files:
            source_path = os.path.join(source_dir, csv_file)
            dest_path = os.path.join(year_dir, csv_file)
            
            try:
                # Copy file to year folder
                shutil.copy2(source_path, dest_path)
                print(f"  Copied: {csv_file}")
                total_moved += 1
                
            except Exception as e:
                error_msg = f"Error copying {csv_file}: {e}"
                print(f"  {error_msg}")
                errors.append(error_msg)
    
    # Summary
    print(f"\n{'='*50}")
    print(f"Sorting complete!")
    print(f"Years processed: {len(sorted_years)} ({min(sorted_years) if sorted_years else 'N/A'} - {max(sorted_years) if sorted_years else 'N/A'})")
    print(f"Total files copied: {total_moved}")
    
    if errors:
        print(f"Errors encountered: {len(errors)}")
        for error in errors[:5]:  # Show first 5 errors
            print(f"  - {error}")
        if len(errors) > 5:
            print(f"  ... and {len(errors) - 5} more errors")
    
    # Show folder structure
    print(f"\nCreated folder structure:")
    for year in sorted_years:
        year_dir = os.path.join(dest_dir, str(year))
        file_count = len([f for f in os.listdir(year_dir) if f.endswith('.csv')])
        print(f"  {year}/: {file_count} files")

# Run the sorting function
sort_csvs_by_year()

Found 8864 CSV files to sort...

Processing year 2001 (220 files):
  Copied: 05_26_2001.csv
  Copied: 05_27_2001.csv
  Copied: 05_28_2001.csv
  Copied: 05_29_2001.csv
  Copied: 05_30_2001.csv
  Copied: 05_31_2001.csv
  Copied: 06_01_2001.csv
  Copied: 06_02_2001.csv
  Copied: 06_03_2001.csv
  Copied: 06_04_2001.csv
  Copied: 06_05_2001.csv
  Copied: 06_06_2001.csv
  Copied: 06_07_2001.csv
  Copied: 06_08_2001.csv
  Copied: 06_09_2001.csv
  Copied: 06_10_2001.csv
  Copied: 06_11_2001.csv
  Copied: 06_12_2001.csv
  Copied: 06_13_2001.csv
  Copied: 06_14_2001.csv
  Copied: 06_15_2001.csv
  Copied: 06_16_2001.csv
  Copied: 06_17_2001.csv
  Copied: 06_18_2001.csv
  Copied: 06_19_2001.csv
  Copied: 06_20_2001.csv
  Copied: 06_21_2001.csv
  Copied: 06_22_2001.csv
  Copied: 06_23_2001.csv
  Copied: 06_24_2001.csv
  Copied: 06_25_2001.csv
  Copied: 06_26_2001.csv
  Copied: 06_27_2001.csv
  Copied: 06_28_2001.csv
  Copied: 06_29_2001.csv
  Copied: 06_30_2001.csv
  Copied: 07_01_2001.csv
  Copied

In [ ]:
Put all csvs not in a year folder into a folder called "All"

In [ ]:
import os
import shutil

def move_csvs_to_all_folder():
    """Move all CSV files from LIB/nyiso_csvs that are not in subfolders into nyiso_csvs/all"""
    
    # Define source directory
    source_dir = os.path.join("LIB", "nyiso_csvs")
    
    # Define destination directory  
    dest_dir = os.path.join("nyiso_csvs", "all")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory {source_dir} does not exist!")
        return
    
    # Get all items in the source directory
    items = os.listdir(source_dir)
    
    # Find CSV files that are directly in LIB/nyiso_csvs (not in subfolders)
    csv_files_to_move = []
    subfolders = []
    
    for item in items:
        item_path = os.path.join(source_dir, item)
        
        # Check if it's a directory (subfolder)
        if os.path.isdir(item_path):
            subfolders.append(item)
        # Check if it's a CSV file directly in the main folder
        elif item.endswith('.csv'):
            csv_files_to_move.append(item)
    
    print(f"Found {len(subfolders)} subfolders in {source_dir}: {subfolders}")
    print(f"Found {len(csv_files_to_move)} CSV files to move from {source_dir} to {dest_dir}")
    
    if not csv_files_to_move:
        print("No CSV files found to move. All CSV files might already be in subfolders.")
        return
    
    # Move CSV files to destination folder
    moved_count = 0
    errors = []
    
    for csv_file in csv_files_to_move:
        source_path = os.path.join(source_dir, csv_file)
        dest_path = os.path.join(dest_dir, csv_file)
        
        try:
            # Check if file already exists in destination
            if os.path.exists(dest_path):
                print(f"Warning: {csv_file} already exists in destination, skipping...")
                continue
            
            # Move the file
            shutil.move(source_path, dest_path)
            print(f"Moved: {csv_file}")
            moved_count += 1
            
        except Exception as e:
            error_msg = f"Error moving {csv_file}: {e}"
            print(error_msg)
            errors.append(error_msg)
    
    # Summary
    print(f"\n{'='*50}")
    print(f"Operation complete!")
    print(f"Source directory: {source_dir}")
    print(f"Destination directory: {dest_dir}")
    print(f"Files moved: {moved_count}")
    
    if errors:
        print(f"Errors encountered: {len(errors)}")
        for error in errors:
            print(f"  - {error}")
    
    # Show final count in destination
    if os.path.exists(dest_dir):
        final_count = len([f for f in os.listdir(dest_dir) if f.endswith('.csv')])
        print(f"Total CSV files now in {dest_dir}: {final_count}")

# Run the function
move_csvs_to_all_folder()



Found 25 subfolders in LIB\nyiso_csvs: ['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Found 0 CSV files to move from LIB\nyiso_csvs to nyiso_csvs\all
No CSV files found to move. All CSV files might already be in subfolders.


In [ ]:
# Delete all CSV files directly inside nyiso_csvs (not in subfolders)

try:
    target_dir = extract_dir  # use existing variable if available
except NameError:
    target_dir = "nyiso_csvs"

if not os.path.exists(target_dir):
    print(f"Error: Directory {target_dir} does not exist!")
else:
    items = os.listdir(target_dir)
    top_level_csvs = [
        f for f in items
        if os.path.isfile(os.path.join(target_dir, f)) and f.lower().endswith(".csv")
    ]

    print(f"Found {len(top_level_csvs)} CSV files in {target_dir} (top level) to delete.")

    deleted = 0
    errors = []

    for fname in top_level_csvs:
        fpath = os.path.join(target_dir, fname)
        try:
            os.remove(fpath)
            print(f"Deleted: {fname}")
            deleted += 1
        except Exception as e:
            msg = f"Error deleting {fname}: {e}"
            print(msg)
            errors.append(msg)

    print(f"\nDeletion complete. Files deleted: {deleted}")
    if errors:
        print(f"Errors encountered: {len(errors)}")
        for e in errors:
            print(f"  - {e}")

In [ ]:
Make a new subfolder of LIB called nyiso_yearly, combine each year's files into one csvs per year 

In [12]:
import os
import pandas as pd
from datetime import datetime

def combine_yearly_csvs():
    """Create LIB/nyiso_yearly folder and combine each year's CSV files into one file per year"""
    
    # Define source and destination directories
    source_dir = os.path.join("LIB", "nyiso_csvs")
    dest_dir = os.path.join("LIB", "nyiso_yearly")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory {source_dir} does not exist!")
        return
    
    # Get all year folders in the source directory
    year_folders = []
    for item in os.listdir(source_dir):
        item_path = os.path.join(source_dir, item)
        if os.path.isdir(item_path) and item.isdigit() and len(item) == 4:
            year_folders.append(item)
    
    year_folders.sort()
    print(f"Found {len(year_folders)} year folders: {year_folders}")
    
    if not year_folders:
        print("No year folders found to process!")
        return
    
    # Process each year
    for year in year_folders:
        year_path = os.path.join(source_dir, year)
        csv_files = [f for f in os.listdir(year_path) if f.endswith('.csv')]
        
        if not csv_files:
            print(f"No CSV files found in {year} folder")
            continue
        
        print(f"\nProcessing year {year} with {len(csv_files)} files...")
        
        # Sort files by date for proper chronological order
        csv_files.sort(key=lambda x: datetime.strptime(x.replace('.csv', '').replace('_', '/'), '%m/%d/%Y'))
        
        # List to store all dataframes for this year
        yearly_dataframes = []
        
        # Read each CSV file
        for csv_file in csv_files:
            file_path = os.path.join(year_path, csv_file)
            try:
                df = pd.read_csv(file_path)
                yearly_dataframes.append(df)
                print(f"  Read: {csv_file} ({len(df)} rows)")
            except Exception as e:
                print(f"  Error reading {csv_file}: {e}")
                continue
        
        # Combine all dataframes for this year
        if yearly_dataframes:
            try:
                combined_df = pd.concat(yearly_dataframes, ignore_index=True)
                
                # Sort by timestamp if timestamp column exists
                if len(combined_df.columns) > 0:
                    timestamp_col = combined_df.columns[0]  # Assume first column is timestamp
                    try:
                        combined_df[timestamp_col] = pd.to_datetime(combined_df[timestamp_col])
                        combined_df = combined_df.sort_values(timestamp_col)
                        combined_df[timestamp_col] = combined_df[timestamp_col].astype(str)
                    except:
                        pass  # Keep original format if datetime conversion fails
                
                # Save combined file
                output_filename = f"{year}_combined.csv"
                output_path = os.path.join(dest_dir, output_filename)
                combined_df.to_csv(output_path, index=False)
                
                print(f"  ✅ Created: {output_filename} ({len(combined_df)} total rows)")
                
            except Exception as e:
                print(f"  Error combining files for {year}: {e}")
        else:
            print(f"  No valid data found for {year}")
    
    # Summary
    print(f"\n{'='*60}")
    print(f"Yearly combination complete!")
    print(f"Output directory: {dest_dir}")
    
    # Show final structure
    if os.path.exists(dest_dir):
        output_files = [f for f in os.listdir(dest_dir) if f.endswith('.csv')]
        output_files.sort()
        print(f"Created {len(output_files)} yearly files:")
        
        for file in output_files:
            file_path = os.path.join(dest_dir, file)
            try:
                df = pd.read_csv(file_path)
                print(f"  {file}: {len(df):,} rows")
            except:
                print(f"  {file}: Error reading file")

# Run the function
combine_yearly_csvs()

Found 25 year folders: ['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

Processing year 2001 with 220 files...
  Read: 05_26_2001.csv (3680 rows)
  Read: 05_27_2001.csv (3750 rows)
  Read: 05_28_2001.csv (3850 rows)
  Read: 05_29_2001.csv (3860 rows)
  Read: 05_30_2001.csv (3700 rows)
  Read: 05_31_2001.csv (3560 rows)
  Read: 06_01_2001.csv (3700 rows)
  Read: 06_02_2001.csv (3990 rows)
  Read: 06_03_2001.csv (3800 rows)
  Read: 06_04_2001.csv (3530 rows)
  Read: 06_05_2001.csv (4000 rows)
  Read: 06_06_2001.csv (3970 rows)
  Read: 06_07_2001.csv (3900 rows)
  Read: 06_08_2001.csv (3740 rows)
  Read: 06_09_2001.csv (3630 rows)
  Read: 06_10_2001.csv (3720 rows)
  Read: 06_11_2001.csv (3870 rows)
  Read: 06_12_2001.csv (3810 rows)
  Read: 06_13_2001.csv (3750 rows)
  Read: 06_14_2001.csv (3980 rows)
  Read: 06_15_2001.csv (4030 rows)
 

Make a new subfolder of LIB called nyiso_all, combine all csvs


In [17]:

import os
import pandas as pd
from datetime import datetime

def combine_all_csvs():
    """Create LIB/nyiso_all folder and combine all CSV files into one master file"""
    
    # Define source and destination directories
    source_dir = os.path.join("LIB", "nyiso_csvs")
    dest_dir = os.path.join("LIB", "nyiso_all")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory {source_dir} does not exist!")
        return
    
    # Collect all CSV files from all subdirectories
    all_csv_files = []
    
    # Walk through all subdirectories
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                all_csv_files.append(file_path)
    
    print(f"Found {len(all_csv_files)} CSV files to combine")
    
    if not all_csv_files:
        print("No CSV files found to combine!")
        return
    
    # Sort files by date for proper chronological order
    def extract_date_from_filename(filepath):
        try:
            filename = os.path.basename(filepath)
            filename_without_ext = filename.replace('.csv', '')
            date_parts = filename_without_ext.split('_')
            if len(date_parts) == 3:
                month, day, year = date_parts
                return datetime.strptime(f"{month}/{day}/{year}", '%m/%d/%Y')
            return datetime.min  # Fallback for files that don't match pattern
        except:
            return datetime.min
    
    all_csv_files.sort(key=extract_date_from_filename)
    
    # List to store all dataframes
    all_dataframes = []
    processed_count = 0
    error_count = 0
    
    print("Reading and combining CSV files...")
    
    # Read each CSV file
    for file_path in all_csv_files:
        try:
            df = pd.read_csv(file_path)
            all_dataframes.append(df)
            processed_count += 1
            
            # Print progress every 100 files
            if processed_count % 100 == 0:
                print(f"  Processed {processed_count}/{len(all_csv_files)} files...")
                
        except Exception as e:
            print(f"  Error reading {os.path.basename(file_path)}: {e}")
            error_count += 1
            continue
    
    # Combine all dataframes
    if all_dataframes:
        try:
            print("Combining all dataframes...")
            combined_df = pd.concat(all_dataframes, ignore_index=True)
            
            # Sort by timestamp if timestamp column exists
            if len(combined_df.columns) > 0:
                timestamp_col = combined_df.columns[0]  # Assume first column is timestamp
                try:
                    print("Sorting by timestamp...")
                    combined_df[timestamp_col] = pd.to_datetime(combined_df[timestamp_col])
                    combined_df = combined_df.sort_values(timestamp_col)
                    combined_df[timestamp_col] = combined_df[timestamp_col].astype(str)
                    print("Timestamp sorting complete.")

_IncompleteInputError: incomplete input (3348982548.py, line 88)

Visualize 
#create new folders 

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

def plot_yearly_data():
    """Create line plots for each year's data in LIB/nyiso_yearly and save to FIGURES folder"""
    
    # Define the directory containing yearly CSV files
    yearly_dir = os.path.join("LIB", "nyiso_yearly")
    
    # Create FIGURES directory if it doesn't exist
    figures_dir = "FIGURES"
    os.makedirs(figures_dir, exist_ok=True)
    
    # Check if directory exists
    if not os.path.exists(yearly_dir):
        print(f"Error: Directory {yearly_dir} does not exist!")
        return
    
    # Get all CSV files in the yearly directory
    csv_files = [f for f in os.listdir(yearly_dir) if f.endswith('.csv')]
    csv_files.sort()
    
    if not csv_files:
        print("No CSV files found in the yearly directory!")
        return
    
    print(f"Found {len(csv_files)} yearly files to plot")
    
    # Set up the plotting style
    plt.style.use('default')
    sns.set_palette("husl")
    
    # Create subplots - adjust based on number of files
    n_files = len(csv_files)
    cols = 2 if n_files > 1 else 1
    rows = (n_files + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 6*rows))
    if n_files == 1:
        axes = [axes]
    elif rows == 1:
        axes = axes.reshape(1, -1)
    
    # Flatten axes array for easier indexing
    axes_flat = axes.flatten() if n_files > 1 else axes
    
    # Plot each year's data
    for i, csv_file in enumerate(csv_files):
        file_path = os.path.join(yearly_dir, csv_file)
        year = csv_file.split('_')[0]  # Extract year from filename
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)
            print(f"Plotting {csv_file} with {len(df)} rows...")
            
            # Assume first column is timestamp and second column is the main data
            if len(df.columns) >= 2:
                timestamp_col = df.columns[0]
                data_col = df.columns[1]
                
                # Convert timestamp to datetime
                df[timestamp_col] = pd.to_datetime(df[timestamp_col])
                
                # Plot on the appropriate subplot
                ax = axes_flat[i] if n_files > 1 else axes_flat[0]
                
                # Create the line plot
                ax.plot(df[timestamp_col], df[data_col], linewidth=1, alpha=0.8)
                ax.set_title(f'NYISO Data - {year}', fontsize=14, fontweight='bold')
                ax.set_xlabel('Date', fontsize=12)
                ax.set_ylabel(data_col, fontsize=12)
                ax.grid(True, alpha=0.3)
                
                # Rotate x-axis labels for better readability
                ax.tick_params(axis='x', rotation=45)
                
                # Add some statistics to the plot
                mean_val = df[data_col].mean()
                max_val = df[data_col].max()
                min_val = df[data_col].min()
                
                # Add text box with statistics
                stats_text = f'Mean: {mean_val:.2f}\nMax: {max_val:.2f}\nMin: {min_val:.2f}'
                ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
                
            else:
                print(f"Warning: {csv_file} has insufficient columns for plotting")
                
        except Exception as e:
            print(f"Error plotting {csv_file}: {e}")
            continue
    
    # Hide any unused subplots
    if n_files > 1:
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].set_visible(False)
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    
    # Save the individual yearly plots
    yearly_plots_filename = os.path.join(figures_dir, "yearly_individual_plots.png")
    plt.savefig(yearly_plots_filename, dpi=300, bbox_inches='tight')
    print(f"Saved individual yearly plots to: {yearly_plots_filename}")
    plt.close()  # Close the figure to free memory
    
    # Create a combined plot showing all years
    plt.figure(figsize=(15, 8))
    
    colors = plt.cm.tab10(range(len(csv_files)))
    
    for i, csv_file in enumerate(csv_files):
        file_path = os.path.join(yearly_dir, csv_file)
        year = csv_file.split('_')[0]
        
        try:
            df = pd.read_csv(file_path)
            
            if len(df.columns) >= 2:
                timestamp_col = df.columns[0]
                data_col = df.columns[1]
                
                df[timestamp_col] = pd.to_datetime(df[timestamp_col])
                
                # Sample data if too many points (for better performance)
                if len(df) > 1000:
                    df_sample = df.sample(n=1000).sort_values(timestamp_col)
                else:
                    df_sample = df
                
                plt.plot(df_sample[timestamp_col], df_sample[data_col], 
                        label=year, linewidth=1.5, alpha=0.8, color=colors[i])
                
        except Exception as e:
            print(f"Error in combined plot for {csv_file}: {e}")
            continue
    
    plt.title('NYISO Data - All Years Comparison', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=14)
    plt.ylabel('Value', fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Save the combined plot
    combined_plot_filename = os.path.join(figures_dir, "all_years_combined_plot.png")
    plt.savefig(combined_plot_filename, dpi=300, bbox_inches='tight')
    print(f"Saved combined plot to: {combined_plot_filename}")
    plt.close()  # Close the figure to free memory
    
    # Create individual plots for each year and save them separately
    for csv_file in csv_files:
        file_path = os.path.join(yearly_dir, csv_file)
        year = csv_file.split('_')[0]
        
        try:
            df = pd.read_csv(file_path)
            
            if len(df.columns) >= 2:
                timestamp_col = df.columns[0]
                data_col = df.columns[1]
                
                # Convert timestamp to datetime
                df[timestamp_col] = pd.to_datetime(df[timestamp_col])
                
                # Create individual plot
                plt.figure(figsize=(12, 6))
                plt.plot(df[timestamp_col], df[data_col], linewidth=1, alpha=0.8, color='blue')
                plt.title(f'NYISO Data - {year}', fontsize=16, fontweight='bold')
                plt.xlabel('Date', fontsize=14)
                plt.ylabel(data_col, fontsize=14)
                plt.grid(True, alpha=0.3)
                plt.xticks(rotation=45)
                
                # Add statistics
                mean_val = df[data_col].mean()
                max_val = df[data_col].max()
                min_val = df[data_col].min()
                
                stats_text = f'Mean: {mean_val:.2f}\nMax: {max_val:.2f}\nMin: {min_val:.2f}'
                plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, 
                        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
                
                plt.tight_layout()
                
                # Save individual year plot
                individual_plot_filename = os.path.join(figures_dir, f"nyiso_data_{year}.png")
                plt.savefig(individual_plot_filename, dpi=300, bbox_inches='tight')
                print(f"Saved {year} plot to: {individual_plot_filename}")
                plt.close()  # Close the figure to free memory
                
        except Exception as e:
            print(f"Error creating individual plot for {csv_file}: {e}")
            continue
    
    print(f"\n{'='*60}")
    print(f"All plots saved to FIGURES directory!")
    print(f"Created {len(csv_files) + 2} plot files:")
    print(f"  - yearly_individual_plots.png (all years in subplots)")
    print(f"  - all_years_combined_plot.png (all years overlaid)")
    for csv_file in csv_files:
        year = csv_file.split('_')[0]
        print(f"  - nyiso_data_{year}.png (individual year plot)")

# Run the plotting function
plot_yearly_data()

Found 25 yearly files to plot
Plotting 2001_combined.csv with 852570 rows...


animation

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from datetime import datetime
import numpy as np
from collections import defaultdict

def create_load_animation():
    """Create animation showing how Load changes over time by Name (zone)"""
    
    # Define source directory and output
    source_dir = os.path.join("LIB", "nyiso_csvs")
    figures_dir = "FIGURES"
    os.makedirs(figures_dir, exist_ok=True)
    
    print("Collecting all CSV files...")
    
    # Collect all CSV files from all year folders
    all_csv_files = []
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                all_csv_files.append(file_path)
    
    # Sort files by date
    def extract_date_from_filename(filepath):
        try:
            filename = os.path.basename(filepath)
            filename_without_ext = filename.replace('.csv', '')
            date_parts = filename_without_ext.split('_')
            if len(date_parts) == 3:
                month, day, year = date_parts
                return datetime.strptime(f"{month}/{day}/{year}", '%m/%d/%Y')
            return datetime.min
        except:
            return datetime.min
    
    all_csv_files.sort(key=extract_date_from_filename)
    
    print(f"Found {len(all_csv_files)} CSV files")
    
    # Sample files for animation (use every Nth file to reduce computation)
    sample_interval = max(1, len(all_csv_files) // 100)  # Use ~100 frames max
    sampled_files = all_csv_files[::sample_interval]
    
    print(f"Using {len(sampled_files)} files for animation")
    
    # Collect all unique zone names first
    all_zones = set()
    print("Identifying all zones...")
    
    for i, file_path in enumerate(sampled_files[:10]):  # Check first 10 files for zones
        try:
            df = pd.read_csv(file_path)
            if 'Name' in df.columns:
                zones = df['Name'].dropna().unique()
                all_zones.update(zones)
        except Exception as e:
            continue
    
    all_zones = sorted(list(all_zones))
    print(f"Found zones: {all_zones}")
    
    # Prepare data for animation
    animation_data = []
    dates = []
    
    print("Processing files for animation...")
    
    for i, file_path in enumerate(sampled_files):
        try:
            df = pd.read_csv(file_path)
            
            if 'Name' in df.columns and 'Load' in df.columns and 'Time Stamp' in df.columns:
                # Get date from filename
                date = extract_date_from_filename(file_path)
                dates.append(date)
                
                # Aggregate load by zone for this date
                zone_loads = {}
                for zone in all_zones:
                    zone_data = df[df['Name'] == zone]['Load']
                    if len(zone_data) > 0:
                        # Use mean load for the day
                        zone_loads[zone] = zone_data.mean()
                    else:
                        zone_loads[zone] = 0
                
                animation_data.append(zone_loads)
                
                if (i + 1) % 10 == 0:
                    print(f"Processed {i + 1}/{len(sampled_files)} files")
                    
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue
    
    if not animation_data:
        print("No valid data found for animation!")
        return
    
    print(f"Creating animation with {len(animation_data)} frames...")
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Define colors for each zone
    colors = plt.cm.tab10(np.linspace(0, 1, len(all_zones)))
    zone_colors = dict(zip(all_zones, colors))
    
    # Initialize bars
    bars = ax.bar(range(len(all_zones)), [0] * len(all_zones), 
                  color=[zone_colors[zone] for zone in all_zones])
    
    # Customize the plot
    ax.set_xlabel('NYISO Zones', fontsize=12)
    ax.set_ylabel('Load (MW)', fontsize=12)
    ax.set_title('NYISO Load by Zone Over Time', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(all_zones)))
    ax.set_xticklabels(all_zones, rotation=45, ha='right')
    
    # Set y-axis limits based on data
    max_load = max(max(frame.values()) for frame in animation_data if frame.values())
    ax.set_ylim(0, max_load * 1.1)
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    # Date text
    date_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, 
                       fontsize=12, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    def animate(frame_num):
        """Animation function"""
        if frame_num < len(animation_data) and frame_num < len(dates):
            frame_data = animation_data[frame_num]
            frame_date = dates[frame_num]
            
            # Update bar heights
            for i, zone in enumerate(all_zones):
                bars[i].set_height(frame_data.get(zone, 0))
            
            # Update date
            date_text.set_text(f'Date: {frame_date.strftime("%Y-%m-%d")}')
            
            return bars + [date_text]
        return bars + [date_text]
    
    # Create animation
    print("Generating animation...")
    anim = animation.FuncAnimation(fig, animate, frames=len(animation_data),
                                 interval=200, blit=False, repeat=True)
    
    # Save animation
    output_path = os.path.join(figures_dir, "nyiso_load_animation.gif")
    
    try:
        # Try to save as GIF
        anim.save(output_path, writer='pillow', fps=5, dpi=100)
        print(f"Animation saved as: {output_path}")
    except Exception as e:
        print(f"Error saving GIF: {e}")
        # Try saving as MP4 instead
        try:
            output_path = os.path.join(figures_dir, "nyiso_load_animation.mp4")
            anim.save(output_path, writer='ffmpeg', fps=5, dpi=100)
            print(f"Animation saved as: {output_path}")
        except Exception as e2:
            print(f"Error saving MP4: {e2}")
            print("Please install pillow or ffmpeg for animation export")
    
    # Also create a static plot showing the final frame
    plt.figure(figsize=(12, 8))
    final_frame = animation_data[-1]
    bars = plt.bar(range(len(all_zones)), [final_frame.get(zone, 0) for zone in all_zones],
                   color=[zone_colors[zone] for zone in all_zones])
    
    plt.xlabel('NYISO Zones', fontsize=12)
    plt.ylabel('Load (MW)', fontsize=12)
    plt.title(f'NYISO Load by Zone - Final Frame ({dates[-1].strftime("%Y-%m-%d")})', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(all_zones)), all_zones, rotation=45, ha='right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    static_path = os.path.join(figures_dir, "nyiso_load_final_frame.png")
    plt.savefig(static_path, dpi=300, bbox_inches='tight')
    print(f"Final frame saved as: {static_path}")
    plt.close()
    
    # Create a line plot showing load trends over time
    plt.figure(figsize=(15, 8))
    
    # Prepare data for line plot
    dates_array = np.array(dates)
    for zone in all_zones[:5]:  # Show top 5 zones to avoid overcrowding
        zone_loads = [frame.get(zone, 0) for frame in animation_data]
        plt.plot(dates_array, zone_loads, label=zone, linewidth=2, alpha=0.8)
    
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Load (MW)', fontsize=12)
    plt.title('NYISO Load Trends by Zone Over Time', fontsize=14, fontweight='bold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    trends_path = os.path.join(figures_dir, "nyiso_load_trends.png")
    plt.savefig(trends_path, dpi=300, bbox_inches='tight')
    print(f"Load trends plot saved as: {trends_path}")
    plt.close()
    
    print(f"\n{'='*60}")
    print("Animation creation complete!")
    print(f"Files created in {figures_dir}:")
    print(f"  - nyiso_load_animation.gif (or .mp4)")
    print(f"  - nyiso_load_final_frame.png")
    print(f"  - nyiso_load_trends.png")

# Run the animation creation
create_load_animation()

In [ ]:
import os
import pandas as pd
from datetime import datetime
import numpy as np

def combine_hourly_csvs():
    """Create LIB/nyiso_hourly folder and combine all CSV files by hour, organized by year"""
    
    # Define source and destination directories
    source_dir = os.path.join("LIB", "nyiso_csvs")
    dest_dir = os.path.join("LIB", "nyiso_hourly")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory {source_dir} does not exist!")
        return
    
    print("Collecting all CSV files for hourly aggregation...")
    
    # Collect all CSV files from all year folders
    all_csv_files = []
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                all_csv_files.append(file_path)
    
    print(f"Found {len(all_csv_files)} CSV files to process")
    
    # Dictionary to store hourly data by year
    yearly_hourly_data = {}
    processed_files = 0
    
    for file_path in all_csv_files:
        try:
            df = pd.read_csv(file_path)
            
            if 'Time Stamp' in df.columns and 'Name' in df.columns and 'Load' in df.columns:
                # Convert timestamp to datetime
                df['Time Stamp'] = pd.to_datetime(df['Time Stamp'])
                
                # Extract year from timestamp
                df['Year'] = df['Time Stamp'].dt.year
                df['Hour'] = df['Time Stamp'].dt.floor('H')  # Round down to hour
                
                # Group by year, hour, and name, then aggregate load
                hourly_agg = df.groupby(['Year', 'Hour', 'Name'])['Load'].agg(['mean', 'max', 'min', 'count']).reset_index()
                hourly_agg.columns = ['Year', 'Time Stamp', 'Name', 'Load_Mean', 'Load_Max', 'Load_Min', 'Data_Points']
                
                # Organize by year
                for year in hourly_agg['Year'].unique():
                    year_data = hourly_agg[hourly_agg['Year'] == year].copy()
                    year_data = year_data.drop('Year', axis=1)  # Remove year column as it's redundant
                    
                    if year not in yearly_hourly_data:
                        yearly_hourly_data[year] = []
                    yearly_hourly_data[year].append(year_data)
                
                processed_files += 1
                if processed_files % 100 == 0:
                    print(f"Processed {processed_files}/{len(all_csv_files)} files...")
                    
        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")
            continue
    
    # Combine and save data for each year
    print("\nCombining and saving hourly data by year...")
    
    for year in sorted(yearly_hourly_data.keys()):
        year_dir = os.path.join(dest_dir, str(year))
        os.makedirs(year_dir, exist_ok=True)
        
        # Combine all hourly data for this year
        year_combined = pd.concat(yearly_hourly_data[year], ignore_index=True)
        
        # Sort by timestamp and name
        year_combined = year_combined.sort_values(['Time Stamp', 'Name'])
        
        # Save to CSV
        output_path = os.path.join(year_dir, f"{year}_hourly_combined.csv")
        year_combined.to_csv(output_path, index=False)
        
        print(f"Created {year}_hourly_combined.csv with {len(year_combined)} hourly records")
    
    print(f"\n{'='*60}")
    print(f"Hourly aggregation complete!")
    print(f"Output directory: {dest_dir}")
    print(f"Years processed: {sorted(yearly_hourly_data.keys())}")

def combine_daily_csvs():
    """Create LIB/nyiso_daily folder and combine all CSV files by day, organized by year"""
    
    # Define source and destination directories
    source_dir = os.path.join("LIB", "nyiso_csvs")
    dest_dir = os.path.join("LIB", "nyiso_daily")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory {source_dir} does not exist!")
        return
    
    print("Collecting all CSV files for daily aggregation...")
    
    # Collect all CSV files from all year folders
    all_csv_files = []
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                all_csv_files.append(file_path)
    
    print(f"Found {len(all_csv_files)} CSV files to process")
    
    # Dictionary to store daily data by year
    yearly_daily_data = {}
    processed_files = 0
    
    for file_path in all_csv_files:
        try:
            df = pd.read_csv(file_path)
            
            if 'Time Stamp' in df.columns and 'Name' in df.columns and 'Load' in df.columns:
                # Convert timestamp to datetime
                df['Time Stamp'] = pd.to_datetime(df['Time Stamp'])
                
                # Extract year and date
                df['Year'] = df['Time Stamp'].dt.year
                df['Date'] = df['Time Stamp'].dt.date
                
                # Group by year, date, and name, then aggregate load
                daily_agg = df.groupby(['Year', 'Date', 'Name'])['Load'].agg([
                    'mean', 'max', 'min', 'std', 'count'
                ]).reset_index()
                daily_agg.columns = ['Year', 'Date', 'Name', 'Load_Mean', 'Load_Max', 'Load_Min', 'Load_Std', 'Data_Points']
                
                # Convert date back to string for consistency
                daily_agg['Date'] = daily_agg['Date'].astype(str)
                
                # Organize by year
                for year in daily_agg['Year'].unique():
                    year_data = daily_agg[daily_agg['Year'] == year].copy()
                    year_data = year_data.drop('Year', axis=1)  # Remove year column as it's redundant
                    
                    if year not in yearly_daily_data:
                        yearly_daily_data[year] = []
                    yearly_daily_data[year].append(year_data)
                
                processed_files += 1
                if processed_files % 100 == 0:
                    print(f"Processed {processed_files}/{len(all_csv_files)} files...")
                    
        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")
            continue
    
    # Combine and save data for each year
    print("\nCombining and saving daily data by year...")
    
    for year in sorted(yearly_daily_data.keys()):
        year_dir = os.path.join(dest_dir, str(year))
        os.makedirs(year_dir, exist_ok=True)
        
        # Combine all daily data for this year
        year_combined = pd.concat(yearly_daily_data[year], ignore_index=True)
        
        # Remove duplicates that might occur from overlapping data
        year_combined = year_combined.drop_duplicates(subset=['Date', 'Name'], keep='first')
        
        # Sort by date and name
        year_combined = year_combined.sort_values(['Date', 'Name'])
        
        # Save to CSV
        output_path = os.path.join(year_dir, f"{year}_daily_combined.csv")
        year_combined.to_csv(output_path, index=False)
        
        print(f"Created {year}_daily_combined.csv with {len(year_combined)} daily records")
    
    print(f"\n{'='*60}")
    print(f"Daily aggregation complete!")
    print(f"Output directory: {dest_dir}")
    print(f"Years processed: {sorted(yearly_daily_data.keys())}")

def create_aggregated_data():
    """Run both hourly and daily aggregation functions"""
    print("Starting data aggregation process...")
    print("\n" + "="*60)
    print("HOURLY AGGREGATION")
    print("="*60)
    combine_hourly_csvs()
    
    print("\n" + "="*60)
    print("DAILY AGGREGATION")
    print("="*60)
    combine_daily_csvs()
    
    print("\n" + "="*60)
    print("ALL AGGREGATIONS COMPLETE!")
    print("="*60)
    print("Created folders:")
    print("  - LIB/nyiso_hourly/[YEAR]/[YEAR]_hourly_combined.csv")
    print("  - LIB/nyiso_daily/[YEAR]/[YEAR]_daily_combined.csv")

# Run the aggregation functions
create_aggregated_data()